# ann hazard score predictor -- 18 classes
**SDG 12.4 | Predictive Analysis Project**  
tabular dataset updated with Refrigerator and Air-Conditioner profiles

## imports

In [1]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, accuracy_score, classification_report,
                              confusion_matrix, f1_score)

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


device: cuda


## configuration

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ANN_DIR    = PROJECT_ROOT / "models" / "ann"
GRAPHS_DIR = ANN_DIR / "graphs"
ANN_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "batch_size"  : 64,
    "num_epochs"  : 200,
    "lr"          : 1e-3,
    "weight_decay": 1e-4,
    "patience"    : 25,
    "dropout_rate": 0.3,
}


## tabular dataset -- 18 component profiles

In [3]:
# hazard profiles derived from:
# global e-waste monitor 2024 (itu/unu), basel convention annex i/ii,
# european weee directive, eu regulation on fluorinated gases (f-gas regulation),
# montreal protocol (CFC/HCFC phase-out schedule)

PROFILES = {
    "Battery"           : {"hazard_base": 90, "contains_lithium":1, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":1, "contains_cfc":0,
                            "recyclable":0, "material_type":"electrochemical", "weight_class":"light"},
    "PCB"               : {"hazard_base": 85, "contains_lithium":0, "contains_lead":1,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"composite", "weight_class":"light"},
    "Mobile"            : {"hazard_base": 80, "contains_lithium":1, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"composite", "weight_class":"light"},
    "Television"        : {"hazard_base": 82, "contains_lithium":0, "contains_lead":1,
                            "contains_mercury":1, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":0, "material_type":"composite", "weight_class":"heavy"},
    "Laptop"            : {"hazard_base": 78, "contains_lithium":1, "contains_lead":1,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"composite", "weight_class":"medium"},
    "light bulbs"       : {"hazard_base": 75, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":1, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":0, "material_type":"glass", "weight_class":"light"},
    "Refrigerator"      : {"hazard_base": 88, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":1,
                            "recyclable":1, "material_type":"metal_cfc", "weight_class":"heavy"},
    "Air-Conditioner"   : {"hazard_base": 85, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":1,
                            "recyclable":1, "material_type":"metal_cfc", "weight_class":"medium"},
    "Microwave"         : {"hazard_base": 60, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"metal", "weight_class":"heavy"},
    "Washing Machine"   : {"hazard_base": 45, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"metal", "weight_class":"heavy"},
    "Printer"           : {"hazard_base": 55, "contains_lithium":0, "contains_lead":1,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"composite", "weight_class":"medium"},
    "Microchip-IC"      : {"hazard_base": 62, "contains_lithium":0, "contains_lead":1,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"silicon", "weight_class":"light"},
    "Keyboard"          : {"hazard_base": 20, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"plastic", "weight_class":"light"},
    "Mouse"             : {"hazard_base": 18, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"plastic", "weight_class":"light"},
    "Resistor"          : {"hazard_base": 15, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"ceramic", "weight_class":"light"},
    "transistor"        : {"hazard_base": 20, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"semiconductor", "weight_class":"light"},
    "heat-sink"         : {"hazard_base": 10, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"metal", "weight_class":"light"},
    "Passive-Component" : {"hazard_base": 22, "contains_lithium":0, "contains_lead":0,
                            "contains_mercury":0, "contains_cadmium":0, "contains_cfc":0,
                            "recyclable":1, "material_type":"semiconductor", "weight_class":"light"},
}

rows = []
N_PER_CLASS = 150

for component, p in PROFILES.items():
    for _ in range(N_PER_CLASS):
        age   = np.random.uniform(0.5, 12)
        wt    = np.random.uniform(0.01, 60)  # refrigerators can be 50-100kg
        cond  = np.random.choice(["working","damaged","broken"], p=[0.25,0.45,0.30])
        reg   = np.random.choice(["low","medium","high"],        p=[0.25,0.45,0.30])
        disp  = np.random.choice(["formal","informal","none"],   p=[0.30,0.35,0.35])

        score = np.clip(
            p["hazard_base"] + np.random.normal(0, 4)
            + min(age * 1.2, 12)
            + {"working":-5, "damaged":2, "broken":8}[cond]
            + {"low":-4, "medium":0, "high":6}[reg]
            + {"formal":-3, "informal":4, "none":2}[disp]
            + p["contains_lithium"]  * 8
            + p["contains_lead"]     * 10
            + p["contains_mercury"]  * 12
            + p["contains_cadmium"]  * 9
            + p["contains_cfc"]      * 15,  # cfc bonus -- highest hazard weighting
            0, 100
        )

        rows.append({
            "component": component, "age_years": round(age, 2),
            "weight_kg": round(wt, 2),
            "contains_lithium": p["contains_lithium"],
            "contains_lead"   : p["contains_lead"],
            "contains_mercury": p["contains_mercury"],
            "contains_cadmium": p["contains_cadmium"],
            "contains_cfc"    : p["contains_cfc"],
            "recyclable": p["recyclable"], "material_type": p["material_type"],
            "weight_class": p["weight_class"], "condition": cond,
            "region_risk": reg, "disposal_history": disp,
            "hazard_score": round(score, 2),
        })

df = pd.DataFrame(rows)
df.to_csv(ANN_DIR / "ewaste_tabular_18cls.csv", index=False)
print(f"dataset: {df.shape[0]} rows x {df.shape[1]} cols")
print(df["hazard_score"].describe().round(2))


dataset: 2700 rows x 15 cols
count    2700.00
mean       69.40
std        31.67
min         0.00
25%        36.33
50%        80.10
75%       100.00
max       100.00
Name: hazard_score, dtype: float64


## preprocessing

In [4]:
le_comp  = LabelEncoder()
le_mat   = LabelEncoder()
le_wc    = LabelEncoder()
le_cond  = LabelEncoder()
le_reg   = LabelEncoder()
le_disp  = LabelEncoder()

df["comp_enc"]  = le_comp.fit_transform(df["component"])
df["mat_enc"]   = le_mat.fit_transform(df["material_type"])
df["wc_enc"]    = le_wc.fit_transform(df["weight_class"])
df["cond_enc"]  = le_cond.fit_transform(df["condition"])
df["reg_enc"]   = le_reg.fit_transform(df["region_risk"])
df["disp_enc"]  = le_disp.fit_transform(df["disposal_history"])

FEATURES = [
    "comp_enc", "age_years", "weight_kg",
    "contains_lithium", "contains_lead", "contains_mercury",
    "contains_cadmium", "contains_cfc",
    "recyclable", "mat_enc", "wc_enc", "cond_enc", "reg_enc", "disp_enc"
]
TARGET = "hazard_score"

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

y_cls = pd.cut(y, bins=[0, 40, 70, 101], labels=["LOW","MEDIUM","HIGH"]).astype(str)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30,
                                              random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42)

sc = StandardScaler()
X_tr  = sc.fit_transform(X_tr)
X_val = sc.transform(X_val)
X_te  = sc.transform(X_te)

print(f"train:{len(X_tr)} | val:{len(X_val)} | test:{len(X_te)}")
print(f"features ({len(FEATURES)}): {FEATURES}")


train:1890 | val:405 | test:405
features (14): ['comp_enc', 'age_years', 'weight_kg', 'contains_lithium', 'contains_lead', 'contains_mercury', 'contains_cadmium', 'contains_cfc', 'recyclable', 'mat_enc', 'wc_enc', 'cond_enc', 'reg_enc', 'disp_enc']


## ann model, training and evaluation

In [5]:
class HazardANN(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(dropout * 0.8),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True), nn.Dropout(dropout * 0.5),
            nn.Linear(64, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.network(x)


class TabDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


tr_dl  = DataLoader(TabDS(X_tr, y_tr),   batch_size=CONFIG["batch_size"], shuffle=True,  drop_last=True)
val_dl = DataLoader(TabDS(X_val, y_val), batch_size=CONFIG["batch_size"], shuffle=False)
te_dl  = DataLoader(TabDS(X_te, y_te),   batch_size=CONFIG["batch_size"], shuffle=False)

model     = HazardANN(len(FEATURES), CONFIG["dropout_rate"]).to(device)
criterion = nn.HuberLoss(delta=5.0)
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=10, factor=0.5)

best_val_loss = float("inf")
best_weights  = None
patience_ctr  = 0
history       = {"train":[], "val":[]}

for epoch in range(1, CONFIG["num_epochs"] + 1):
    model.train()
    tl = 0
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item() * len(xb)
    tl /= len(TabDS(X_tr, y_tr))

    model.eval()
    vl = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            vl += criterion(model(xb.to(device)), yb.to(device)).item() * len(xb)
    vl /= len(TabDS(X_val, y_val))

    scheduler.step(vl)
    history["train"].append(tl)
    history["val"].append(vl)

    if vl < best_val_loss:
        best_val_loss = vl
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(model.state_dict(), ANN_DIR / "ann_best_18cls.pth")
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG["patience"]:
            print(f"early stopping at epoch {epoch}")
            break

    if epoch % 20 == 0:
        print(f"  epoch {epoch:03d} | train {tl:.4f} | val {vl:.4f}")

model.load_state_dict(best_weights)

model.eval()
all_p, all_t = [], []
with torch.no_grad():
    for xb, yb in te_dl:
        all_p.extend(model(xb.to(device)).cpu().squeeze().numpy())
        all_t.extend(yb.squeeze().numpy())

all_p = np.array(all_p)
all_t = np.array(all_t)

mae  = mean_absolute_error(all_t, all_p)
rmse = np.sqrt(mean_squared_error(all_t, all_p))
r2   = r2_score(all_t, all_p)
mape = np.mean(np.abs((all_t - all_p) / (all_t + 1e-8))) * 100

print(f"\ntest results -- ann (18 classes, 14 features):")
print(f"  mae  : {mae:.4f}")
print(f"  rmse : {rmse:.4f}")
print(f"  r2   : {r2:.4f}")
print(f"  mape : {mape:.2f}%")

def to_hcls(s):
    return ["HIGH" if x>=70 else "MEDIUM" if x>=40 else "LOW" for x in s]

cls_acc = accuracy_score(to_hcls(all_t), to_hcls(all_p))
print(f"  hazard class accuracy: {cls_acc:.4f}")
print(classification_report(to_hcls(all_t), to_hcls(all_p),
                             target_names=["HIGH","MEDIUM","LOW"], digits=4))


  epoch 020 | train 23.9551 | val 11.6823
  epoch 040 | train 22.7649 | val 9.5887
  epoch 060 | train 20.2301 | val 9.5528
  epoch 080 | train 21.7269 | val 6.9501
  epoch 100 | train 19.2894 | val 6.4923
  epoch 120 | train 21.3704 | val 6.6391
early stopping at epoch 125

test results -- ann (18 classes, 14 features):
  mae  : 2.7732
  rmse : 3.9278
  r2   : 0.9846
  mape : 7.20%
  hazard class accuracy: 0.9506
              precision    recall  f1-score   support

        HIGH     0.9736    0.9822    0.9779       225
      MEDIUM     0.9313    0.9919    0.9606       123
         LOW     0.8936    0.7368    0.8077        57

    accuracy                         0.9506       405
   macro avg     0.9328    0.9036    0.9154       405
weighted avg     0.9495    0.9506    0.9487       405



## plots

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("ann hazard predictor -- 18-class e-waste", fontsize=13, fontweight="bold")

axes[0].plot(history["train"], color="#2c7bb6", linewidth=1.5, label="train")
axes[0].plot(history["val"],   color="#d7191c", linewidth=1.5, label="val")
axes[0].set_title("huber loss"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].scatter(all_t, all_p, alpha=0.4, s=12, color="#2c7bb6")
axes[1].plot([0,100],[0,100], "r--", linewidth=1.5)
axes[1].set_title(f"predicted vs actual (r2={r2:.4f})")
axes[1].set_xlabel("actual"); axes[1].set_ylabel("predicted"); axes[1].grid(alpha=0.3)

cm = confusion_matrix(to_hcls(all_t), to_hcls(all_p), labels=["HIGH","MEDIUM","LOW"])
sns.heatmap(cm / cm.sum(axis=1)[:,None], ax=axes[2], annot=True, fmt=".3f",
            cmap="Blues", xticklabels=["HIGH","MEDIUM","LOW"],
            yticklabels=["HIGH","MEDIUM","LOW"], linewidths=0.5)
axes[2].set_title(f"hazard class confusion (acc={cls_acc:.4f})")
axes[2].set_ylabel("true"); axes[2].set_xlabel("predicted")

plt.tight_layout()
plt.savefig(GRAPHS_DIR / "ann_results_18cls.png", dpi=150, bbox_inches="tight")
plt.close()
print("ann plots saved")

results = {"model":"HazardANN_18cls", "features":FEATURES,
           "mae":round(float(mae),4), "rmse":round(float(rmse),4),
           "r2":round(float(r2),4), "mape":round(float(mape),2),
           "hazard_class_accuracy":round(float(cls_acc),4)}
with open(ANN_DIR / "ann_results_18cls.json", "w") as f:
    json.dump(results, f, indent=2)
print("ann results saved")


ann plots saved
ann results saved
